# Pipeline Cào Metadata và Trích dẫn arXiv (Milestone 1)

Notebook này phục vụ việc cào **Metadata bài báo (arXiv API)** và **Danh sách trích dẫn (Semantic Scholar API)** theo dải bài báo được phân công của thành viên.

### Luồng hoạt động chính:
1. **Sinh danh sách ID**: Dựa vào `ranges.yaml` hoặc phân chia phần (`part` 1-5).
2. **Cào Metadata**: Sử dụng `arxiv` client với `RateLimiter` (tối thiểu 3.0 giây) lưu vào `data/papers.jsonl`.
3. **Cào References**: Sử dụng Semantic Scholar Graph API với `RateLimiter` lưu vào `data/references.jsonl`.
4. **Cơ chế Resume**: Tự động bỏ qua các bài đã cào thành công khi chạy lại.
5. **Đo lường & Thống kê**: Báo cáo số lượng thành công, lỗi, và các chỉ số phục vụ báo cáo Milestone 1.

## 1. Kết nối Google Drive và Thiết lập Môi trường (trên Colab)

In [ ]:
# Chạy cell này nếu bạn thực thi trên Google Colab
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    # Thay đổi đường dẫn tới thư mục repo trên Google Drive của bạn:
    os.chdir('/content/drive/MyDrive/Colab_Notebooks/arXiv-Project')
    print("Thư mục làm việc hiện tại:", os.getcwd())
except ImportError:
    print("Đang chạy ở môi trường cục bộ (local).")

In [ ]:
# Cài đặt các thư viện cần thiết
%pip install -r requirements.txt

## 2. Sinh danh sách arXiv ID cho dải được phân công

In [ ]:
from src import config
from src.metadata_module import generate_id_list, PARTS

# Chọn phần được phân công (1 đến 5)
PART = 1

id_list = generate_id_list(part=PART, out_path=config.DATA_DIR / "id_list.txt")
print(f"Đã sinh {len(id_list)} ID cho phần {PART}")
print("5 ID đầu tiên:", id_list[:5])
print("5 ID cuối cùng:", id_list[-5:])

## 3. Cào Metadata bài báo từ arXiv API

In [ ]:
from src.metadata_module import crawl_papers

# Chạy cào metadata
# Đặt limit=10 để chạy thử nghiệm, hoặc limit=0 để cào toàn bộ danh sách
stats_papers = crawl_papers(
    id_list_path=config.DATA_DIR / "id_list.txt",
    out_path=config.DATA_DIR / "papers.jsonl",
    limit=10
)
print("Kết quả cào papers:", stats_papers)

## 4. Cào Danh sách Trích dẫn (References) từ Semantic Scholar

In [ ]:
from src.metadata_module import crawl_references

# Chạy cào references
# Đặt limit=10 để chạy thử nghiệm, hoặc limit=0 để cào toàn bộ
stats_refs = crawl_references(
    papers_path=config.DATA_DIR / "papers.jsonl",
    out_path=config.DATA_DIR / "references.jsonl",
    limit=10
)
print("Kết quả cào references:", stats_refs)

## 5. Thống kê & Kiểm tra Tính toàn vẹn Dữ liệu

In [ ]:
import pandas as pd
from src.metadata_module import summarize, load_papers, load_references

# Tổng quan thống kê
summary = summarize()
print("=== BẢNG THỐNG KÊ TỔNG HỢP ===")
for key, val in summary.items():
    print(f"{key:20s}: {val}")

# Hiển thị mẫu dữ liệu bài báo
sample_papers = list(load_papers())[:3]
if sample_papers:
    df = pd.DataFrame(sample_papers)
    display(df[["arxiv_id", "title", "categories", "submission_date", "versions"]])
else:
    print("Chưa có bài báo nào được lưu.")